### Summary

**EAGLE-Net Project Overview:**

✓ **Models Implemented:**
- BaselineCNN: Standard CNN (~500K parameters)
- LightweightCNN: Efficient depthwise separable (~100K parameters)
- EAGLENet: Lightweight + Attention (~150K parameters)

✓ **Training Features:**
- Reproducible train/val/test split (70/15/15)
- Comprehensive metrics (Accuracy, F1, Precision, Recall)
- Model checkpointing and history tracking
- Training visualization (curves, confusion matrix)

✓ **Inference Capabilities:**
- Single image predictions
- Top-k predictions with confidence scores
- Batch processing support

✓ **Key Results:**
- Efficient models suitable for edge deployment
- Attention mechanisms improve spatial relevance
- Fast inference on CPU

### Terminal Commands

**1. Train Model:**
```bash
python train.py --model eager_net --epochs 100 --batch-size 64
```

**2. Evaluate Model:**
```bash
python evaluate.py --split test
```

**3. Run Inference:**
```bash
python inference.py --image-path path/to/image.jpg
```

**4. Launch Streamlit Demo:**
```bash
streamlit run app/streamlit_app.py
```

### Project Structure
- `src/data/` - Dataset loading and preprocessing
- `src/models/` - Model architectures
- `src/training/` - Training loop and utilities
- `src/inference/` - Prediction functions
- `src/utils/` - Metrics and visualization
- `app/` - Streamlit demo application
- `artifacts/` - Saved models and results

### Next Steps

1. **Train Full Model:** Use 100+ epochs for production quality
2. **Hyperparameter Tuning:** Adjust learning rate, batch size, regularization
3. **Data Augmentation:** Experiment with additional transforms
4. **Model Ensemble:** Combine multiple models for better accuracy
5. **Deploy:** Use Streamlit or FastAPI for inference service

## 10. Summary and Next Steps

Key takeaways and usage instructions

In [ ]:
# Visualize prediction
from src.utils.visualization import plot_top_predictions

# Denormalize image for visualization
img_viz = test_image.cpu()
img_viz = img_viz * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
img_viz = torch.clamp(img_viz, 0, 1)
img_array = img_viz.permute(1, 2, 0).numpy()

fig = plot_top_predictions(
    img_array,
    result['probabilities'],
    class_names,
    top_k=3,
)
plt.show()

In [ ]:
# Test inference on a batch from test set
test_batch = next(iter(dataloaders['test']))
test_images, test_labels = test_batch

# Get a single image
test_image = test_images[0]
true_label = test_labels[0]

# Predict
result = predictor.predict_from_array(test_image)

print(f"\n🔮 Prediction Results:")
print(f"  True class: {class_names[true_label]}")
print(f"  Predicted: {result['class_name']}")
print(f"  Confidence: {result['confidence']:.2%}")
print(f"  Correct: {'✓' if result['class_idx'] == true_label else '✗'}")

# Get top-3
top_k_result = predictor.get_top_k(str(DATASET_DIR), k=3)
probs = result['probabilities']
top_indices = np.argsort(probs)[-3:][::-1]

print(f"\n🥇 Top-3 Predictions:")
medals = ['🥇', '🥈', '🥉']
for medal, idx in zip(medals, top_indices):
    print(f"  {medal} {class_names[idx]:.<25} {probs[idx]:.2%}")

In [ ]:
# Import predictor
from src.inference.predictor import Predictor

# Create predictor
predictor = Predictor(trainer.model, class_names, device=DEVICE, img_size=IMG_SIZE)

print("✓ Predictor initialized")
print(f"  Model: EAGLENet")
print(f"  Classes: {NUM_CLASSES}")
print(f"  Device: {DEVICE}")

## 9. Inference on Single Images

Load model and make predictions on new images

In [ ]:
# Plot confusion matrix
from src.utils.visualization import plot_confusion_matrix

cm = test_tracker.get_confusion_matrix()
fig = plot_confusion_matrix(cm, class_names, normalize=True)
plt.show()

print(f"✓ Confusion matrix plotted")

In [ ]:
# Load best model
trainer.load_checkpoint('best_model.pt')

# Evaluate on test set
print("📊 Evaluating on Test Set...\n")
test_metrics, test_tracker = trainer.evaluate(dataloaders['test'], stage='Test')

## 8. Model Evaluation and Confusion Matrix

Evaluate on test set and visualize predictions

In [ ]:
# Plot training curves
from src.utils.visualization import plot_training_curves

fig = plot_training_curves(
    history['train_loss'],
    history['val_loss'],
    history['train_acc'],
    history['val_acc'],
)
plt.show()

print(f"✓ Training curves plotted")

## 7. Training Visualization

Visualize training curves and metrics

In [ ]:
# Train the model (reduced epochs for demonstration)
DEMO_EPOCHS = 10  # Reduced for faster execution in notebook
print(f"\n📚 Training for {DEMO_EPOCHS} epochs...")

history = trainer.fit(num_epochs=DEMO_EPOCHS, verbose=True)

print(f"\n✓ Training complete!")
print(f"  Best validation accuracy: {trainer.history['best_val_acc']:.4f}")
print(f"  Best epoch: {trainer.history['best_epoch']}")

In [ ]:
# Import trainer
from src.training.trainer import Trainer

# Use EAGLENet for demonstration
model = eagle_model

# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-5)

# Create trainer
trainer = Trainer(
    model=model,
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    test_loader=dataloaders['test'],
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    save_dir=SAVE_DIR,
    class_names=class_names,
)

print("✓ Trainer initialized")
print(f"  Model: EAGLENet")
print(f"  Loss: CrossEntropyLoss")
print(f"  Optimizer: Adam (lr=0.001, weight_decay=5e-5)")
print(f"  Epochs: {EPOCHS}")

## 6. Training Loop with Metrics

Define training loop with loss, optimizer, and metric tracking

In [ ]:
# Test forward pass with dummy input
print("\n✓ Testing forward passes...")
dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

with torch.no_grad():
    out_baseline = baseline_model(dummy_input)
    out_lightweight = lightweight_model(dummy_input)
    out_eagle = eagle_model(dummy_input)

print(f"  Input shape: {dummy_input.shape}")
print(f"  Baseline output: {out_baseline.shape}")
print(f"  Lightweight output: {out_lightweight.shape}")
print(f"  EAGLENet output: {out_eagle.shape}")

In [ ]:
# Create EAGLENet
eagle_model = EAGLENet(num_classes=NUM_CLASSES)
eagle_model.to(DEVICE)

print("📊 EAGLENet Architecture:")
print("  Base: LightweightCNN")
print("  + Channel Attention (SE-style)")
print("  + Spatial Attention")
print(f"\nTotal Parameters: {count_parameters(eagle_model):,}")
print(f"Model size: ~{count_parameters(eagle_model) * 4 / (1024**2):.2f} MB")

## 5. EAGLE-Net with Attention Mechanisms

Lightweight CNN enhanced with Channel and Spatial Attention

In [ ]:
# Compare model sizes
baseline_params = count_parameters(baseline_model)
lightweight_params = count_parameters(lightweight_model)
reduction = (1 - lightweight_params / baseline_params) * 100

print(f"\n📈 Model Comparison:")
print(f"  Baseline:     {baseline_params:>10,} parameters")
print(f"  Lightweight:  {lightweight_params:>10,} parameters")
print(f"  Reduction:    {reduction:>10.1f}%")

In [ ]:
# Create LightweightCNN
lightweight_model = LightweightCNN(num_classes=NUM_CLASSES)
lightweight_model.to(DEVICE)

print("📊 LightweightCNN Architecture:")
print(lightweight_model)
print(f"\nTotal Parameters: {count_parameters(lightweight_model):,}")
print(f"Model size: ~{count_parameters(lightweight_model) * 4 / (1024**2):.2f} MB")

## 4. Lightweight CNN with Depthwise Separable Convolutions

Efficient variant using depthwise separable convolutions

In [ ]:
# Import model utilities
from src.models.architectures import BaselineCNN, LightweightCNN, EAGLENet, count_parameters

# Create and display BaselineCNN
baseline_model = BaselineCNN(num_classes=NUM_CLASSES)
baseline_model.to(DEVICE)

print("📊 BaselineCNN Architecture:")
print(baseline_model)
print(f"\nTotal Parameters: {count_parameters(baseline_model):,}")
print(f"Model size: ~{count_parameters(baseline_model) * 4 / (1024**2):.2f} MB")

## 3. Baseline CNN Implementation

Define a simple CNN baseline architecture

In [ ]:
# Count class distribution in training set
class_counts = {cls: 0 for cls in class_names}
for images, labels in dataloaders['train']:
    for label in labels:
        class_counts[class_names[label]] += 1

# Plot class distribution
fig, ax = plt.subplots(figsize=(12, 5))
classes = list(class_counts.keys())
counts = list(class_counts.values())

ax.bar(classes, counts, color='steelblue', edgecolor='navy', alpha=0.7)
ax.set_xlabel('Class', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Samples', fontsize=11, fontweight='bold')
ax.set_title('Training Set Class Distribution', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"✓ Class distribution visualized")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

# Get first batch and denormalize for visualization
train_batch = next(iter(dataloaders['train']))
images, labels = train_batch

# Denormalization values (ImageNet)
mean = torch.tensor([0.485, 0.456, 0.406])
std = torch.tensor([0.229, 0.224, 0.225])

for idx in range(10):
    img = images[idx].cpu()
    # Denormalize
    img = img * std.view(3, 1, 1) + mean.view(3, 1, 1)
    img = torch.clamp(img, 0, 1)
    
    axes[idx].imshow(img.permute(1, 2, 0))
    axes[idx].set_title(f"{class_names[labels[idx]]}", fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Sample Images from Each Class', fontsize=12, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"✓ Displayed {len(images)} sample images")

In [ ]:
# Import dataset utilities
from src.data.dataset import EuroSATDataset, create_dataloaders, EUROSAT_CLASSES, get_data_transforms

# Check if dataset exists
if not Path(DATASET_DIR).exists():
    print(f"❌ Dataset not found at {DATASET_DIR}")
    print(f"Please download EuroSAT and place it in {DATASET_DIR}/")
else:
    # Load dataset metadata
    dataloaders, class_names = create_dataloaders(
        DATASET_DIR,
        batch_size=BATCH_SIZE,
        seed=SEED,
        num_workers=0,
        img_size=IMG_SIZE
    )
    
    print(f"✓ Dataset loaded successfully!")
    print(f"  Classes: {len(class_names)}")
    print(f"  Class names: {', '.join(class_names[:3])}...")  # Show first 3

## 2. Dataset Loading and Exploration

Load EuroSAT dataset and visualize sample images

In [ ]:
# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_DIR = '../data/EuroSAT'
SAVE_DIR = '../artifacts'
BATCH_SIZE = 64
IMG_SIZE = 64
NUM_CLASSES = 10
EPOCHS = 50

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

print(f"✓ Configuration:")
print(f"  Device: {DEVICE}")
print(f"  Seed: {SEED}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}×{IMG_SIZE}")

In [ ]:
# Import required libraries
import sys
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json

# Add src to path
sys.path.insert(0, '..')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. Setup and Environment Configuration

# EAGLE-Net: Efficient Attention for Geo-spatial Land Estimation Network

**A lightweight CNN with channel attention for EuroSAT RGB satellite land-use classification**

This notebook walks through the complete EAGLE-Net pipeline:
- Dataset loading and exploration
- Model architectures (Baseline, Lightweight, EAGLENet)
- Training with metrics tracking
- Evaluation and visualization
- Inference on new images

**Dataset:** EuroSAT RGB (10 classes, 64×64, ~27,000 images)  
**Framework:** PyTorch  
**Goal:** Efficient satellite image classification